# Kindle Data (NLP) CLF


In [2]:
import pandas as pd

data = pd.read_csv("data/all_kindle_review.csv")

In [3]:
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [4]:
data = data[["reviewText", "rating"]]

data.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [5]:
data.shape

(12000, 2)

In [6]:
data.isnull().sum()

,0
reviewText,0
rating,0


In [7]:
data["rating"].unique()

array([3, 5, 4, 2, 1])

In [8]:
data["rating"].value_counts()

,count
rating,
5,3000
4,3000
3,2000
2,2000
1,2000


## Preprocessing


In [9]:
# positive review = 1, negative review = 0
data["rating"] = data["rating"].apply(lambda x: 0 if x < 3 else 1)

In [10]:
data["rating"].value_counts()

,count
rating,
1,8000
0,4000


In [11]:
# lower all the cases
data["reviewText"] = data["reviewText"].str.lower()

In [12]:
from bs4 import BeautifulSoup
import re
from nltk.corpus import stopwords

In [17]:
## Removing special characters
data["reviewText"] = data["reviewText"].apply(
    lambda x: re.sub("[^a-z A-z 0-9-]+", "", x)
)
## Remove the stopswords
data["reviewText"] = data["reviewText"].apply(
    lambda x: " ".join([y for y in x.split() if y not in stopwords.words("english")])
)
## Remove url
data["reviewText"] = data["reviewText"].apply(
    lambda x: re.sub(
        r"(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?",
        "",
        str(x),
    )
)
## Remove html tags
data["reviewText"] = data["reviewText"].apply(
    lambda x: BeautifulSoup(x, "lxml").get_text()
)
## Remove any additional spaces
data["reviewText"] = data["reviewText"].apply(lambda x: " ".join(x.split()))

In [18]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [19]:
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [20]:
data["reviewText"] = data["reviewText"].apply(lambda x: lemmatize_words(x))

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data["reviewText"], data["rating"], test_size=0.2, random_state=42
)

In [22]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()

X_train_bow = bow.fit_transform(X_train).toarray()

X_test_bow = bow.transform(X_test).toarray()

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

X_train_tfidf = tfidf.fit_transform(X_train).toarray()

X_test_tfidf = tfidf.transform(X_test).toarray()

In [38]:
from gensim.models import Word2Vec
import numpy as np

from nltk.tokenize import word_tokenize

X_train_tokens = X_train.apply(lambda x: word_tokenize(str(x).lower()))
X_test_tokens = X_test.apply(lambda x: word_tokenize(str(x).lower()))


# 2. Train Word2Vec on the training tokens
w2v_model = Word2Vec(
    sentences=X_train_tokens.tolist(),
    vector_size=100,  # embedding dimension
    window=5,  # context window size
    min_count=1,  # ignore words with freq < this
    workers=4,  # parallel threads
    sg=1,  # 1 = skip-gram, 0 = CBOW
)


# 3. Function to convert a tokenized sentence into a single vector
#    (average of its word vectors — the standard baseline approach)
def get_avg_vector(tokens, model, vector_size):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)


# 4. Build feature matrices
X_train_w2v = np.array(
    [get_avg_vector(tokens, w2v_model, 100) for tokens in X_train_tokens]
)
X_test_w2v = np.array(
    [get_avg_vector(tokens, w2v_model, 100) for tokens in X_test_tokens]
)

In [45]:
import gensim.downloader as api
import numpy as np
from nltk.tokenize import word_tokenize

X_train_tokens = X_train.apply(lambda x: word_tokenize(str(x).lower()))
X_test_tokens = X_test.apply(lambda x: word_tokenize(str(x).lower()))

# Load Google's pretrained model (this downloads ~1.6GB the first time — takes a few minutes)
google_w2v = api.load("word2vec-google-news-300")

# Note: vector_size is 300 for this model, not 100
def get_avg_vector(tokens, model, vector_size):
    vectors = [model[word] for word in tokens if word in model]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

X_train_google_w2v = np.array(
    [get_avg_vector(tokens, google_w2v, 300) for tokens in X_train_tokens]
)
X_test_google_w2v = np.array(
    [get_avg_vector(tokens, google_w2v, 300) for tokens in X_test_tokens]
)

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [47]:
from sklearn.naive_bayes import GaussianNB

nb_clf_bow = GaussianNB().fit(X_train_bow, y_train)

nb_clf_tfidf = GaussianNB().fit(X_train_tfidf, y_train)

nb_clf_w2v = GaussianNB().fit(X_train_w2v, y_train)

nb_clf_gw2v = GaussianNB().fit(X_train_google_w2v, y_train)

In [48]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [50]:
print("BOW Accuracy:", accuracy_score(y_test, nb_clf_bow.predict(X_test_bow)))

print("TF-IDF Accuracy:", accuracy_score(y_test, nb_clf_tfidf.predict(X_test_tfidf)))

print("Word2Vec Accuracy:", accuracy_score(y_test, nb_clf_w2v.predict(X_test_w2v)))

print("Google Word2Vec Accuracy:", accuracy_score(y_test, nb_clf_gw2v.predict(X_test_google_w2v)))

BOW Accuracy: 0.5745833333333333
TF-IDF Accuracy: 0.57875
Word2Vec Accuracy: 0.75
Google Word2Vec Accuracy: 0.7441666666666666


In [51]:
confusion_matrix(y_test, nb_clf_bow.predict(X_test_bow))

array([[499, 304],
       [717, 880]])

In [52]:
confusion_matrix(y_test, nb_clf_tfidf.predict(X_test_tfidf))

array([[488, 315],
       [696, 901]])

In [53]:
confusion_matrix(y_test, nb_clf_w2v.predict(X_test_w2v))

array([[ 654,  149],
       [ 451, 1146]])

In [54]:
confusion_matrix(y_test, nb_clf_gw2v.predict(X_test_google_w2v))

array([[ 637,  166],
       [ 448, 1149]])